In [ ]:
!pip install PyMuPDF transformers torch sentencepiece reportlab
import fitz # PyMuPDF
import re
import time
from typing import Dict, List
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from transformers import pipeline, BartForConditionalGeneration, BartTokenizer
import torch
import os
import sys

# --- CONFIGURATION ---
PDF_PATH = "/content/covid Protocol.pdf"
FALLBACK_PDF_PATH = "/content/extracted_sections (5).pdf"
OUTPUT_PDF = "summarized_bart.pdf"

BART_MODEL = "facebook/bart-large-cnn"
QA_MODEL = "distilbert-base-cased-distilled-squad"

MAX_INPUT_TOKENS = 900
SUMMARY_MAX_LENGTH = 500
SUMMARY_MIN_LENGTH = 100
SUMMARY_DO_SAMPLE = False
SUMMARY_LENGTH_PENALTY = 2.0
SUMMARY_NUM_BEAMS = 4

ORDERED_HEADERS = [
    ('inclusion_criteria', r"6\.1\s+Inclusion Criteria for All Subjects"),
    ('exclusion_criteria', r"6\.2\s+Exclusion Criteria"),
    ('vaccine_delay', r"6\.3\s+Vaccine Delay\s+Recommendations"),
    ('primary_endpoints', r"(?:\d+\.\d+\s+)?Primary Endpoints"),
    ('secondary_endpoints', r"(?:\d+\.\d+\s+)?Secondary Endpoints"),
    ('_end_secondary_endpoints', r"(?:\d+\.\d+\s+)?Exploratory Endpoints"),
    ('treatment_arms', r"5\.3\s+Trial Population|--- Treatment Arms ---"),
    ('_end_treatment_arms', r"7\.1\s+Dosing Schedule|--- Dosing Schedule ---"),
    ('dosing_schedule', r"7\.1\s+Dosing Schedule|--- Dosing Schedule ---"),
    ('_end_dosing_schedule', r"7\.2\s+Preparation/Handling/Storage/Accountability"),
]

QUERIES_FOR_QA = {
    "inclusion_criteria": [
        "What are the criteria for males and females to be included in the study?",
        "What specific birth control methods are considered highly effective?"
    ],
    "exclusion_criteria": [
        "List all specific medical conditions that exclude a subject from the trial.",
        "What is the exclusion period for other investigational or licensed vaccines?",
        "What are the criteria related to prior SARS-CoV-2 vaccination?",
        "What are the rules regarding the use of immunosuppressants?"
    ],
    "vaccine_delay": [
        "Under what conditions can the second vaccine dose be delayed?",
        "What happens if a subject develops COVID-19 after the first dose?"
    ],
    "primary_endpoints": [
        "What is the primary endpoint of the study?",
        "What is the timeframe for collecting AEs and SAEs for the primary safety endpoint?"
    ],
    "secondary_endpoints": [
        "What are the key secondary efficacy endpoints?",
        "What are the secondary immunogenicity endpoints and their timeframes?"
    ],
    "treatment_arms": [
        "What is the overall design of the trial?",
        "What are the treatment arms and dose levels?"
    ],
    "dosing_schedule": [
        "What is the dosing schedule?",
        "What is the dose level and administration route?",
        "How is the placebo prepared and stored?"
    ]
}

def choose_pdf_path():
    if os.path.exists(PDF_PATH):
        return PDF_PATH
    if os.path.exists(FALLBACK_PDF_PATH):
        return FALLBACK_PDF_PATH
    raise FileNotFoundError(f"Neither {PDF_PATH} nor {FALLBACK_PDF_PATH} exist. Put your PDF path in PDF_PATH variable.")

def pdf_to_text(path: str) -> str:
    doc = fitz.open(path)
    pages = []
    for p in range(len(doc)):
        pages.append(doc.load_page(p).get_text("text"))
    doc.close()
    full = "\n\n".join(pages)
    return re.sub(r'\u00A0', ' ', full)

def remove_page_numbers(section: str) -> str:
    lines = section.splitlines()
    filtered = []
    for line in lines:
        line = line.strip()
        if re.match(r'^Page \d+ of \d+', line):
            continue
        if re.match(r'^.*\.+\s*\d+$', line):
            continue
        if re.match(r'^\d+$', line):
            continue
        filtered.append(line)
    return "\n".join(filtered)

def extract_all_sections_ordered(text: str, ordered_headers: List) -> Dict[str, str]:
    header_matches = []
    for key, pat in ordered_headers:
        for m in re.finditer(pat, text, re.IGNORECASE):
            header_matches.append({'key': key, 'start': m.start(), 'header': m.group(), 'pattern': pat})
    header_matches = sorted(header_matches, key=lambda x: x['start'])

    sections = {}
    for i, match in enumerate(header_matches):
        if match['key'].startswith('_'):
            continue
        start = match['start']
        end_key = f'_end_{match["key"]}'
        end_match = next((h for h in header_matches if h['key'] == end_key and h['start'] > start), None)
        end = end_match['start'] if end_match else len(text)
        section_text = text[start:end].strip()
        section_text = remove_page_numbers(section_text)
        sections[match['key']] = section_text

    for key, _ in ordered_headers:
        if key.startswith('_'):
            continue
        if key not in sections:
            sections[key] = ""

    return sections

def chunk_text_by_sentences(text: str, tokenizer: BartTokenizer, max_tokens: int = MAX_INPUT_TOKENS) -> List[str]:
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    current_chunk_sentences = []
    current_chunk_token_len = 0

    for sentence in sentences:
        if not sentence.strip():
            continue
        sentence_token_len = len(tokenizer.encode(sentence, add_special_tokens=False))

        if current_chunk_token_len + sentence_token_len > max_tokens:
            if current_chunk_sentences:
                chunks.append(" ".join(current_chunk_sentences))
            current_chunk_sentences = [sentence]
            current_chunk_token_len = sentence_token_len
        else:
            current_chunk_sentences.append(sentence)
            current_chunk_token_len += sentence_token_len

    if current_chunk_sentences:
        chunks.append(" ".join(current_chunk_sentences))
    return chunks

def summarize_text_bart(text: str, summarizer_pipeline, tokenizer: BartTokenizer) -> str:
    """Always chunk first. If CUDA fails, retry on CPU automatically."""
    if not text or text.strip() == "":
        return ""

    chunks = chunk_text_by_sentences(text, tokenizer, max_tokens=MAX_INPUT_TOKENS)
    chunk_summaries = []
    for ch in chunks:
        try:
            out = summarizer_pipeline(
                ch.strip(),
                max_length=SUMMARY_MAX_LENGTH,
                min_length=SUMMARY_MIN_LENGTH,
                do_sample=SUMMARY_DO_SAMPLE,
                num_beams=SUMMARY_NUM_BEAMS,
                length_penalty=SUMMARY_LENGTH_PENALTY
            )
        except RuntimeError as e:
            print("CUDA error during summarization, retrying on CPU...")
            summarizer_cpu = pipeline("summarization", model=BART_MODEL, tokenizer=tokenizer, device=-1)
            out = summarizer_cpu(
                ch.strip(),
                max_length=SUMMARY_MAX_LENGTH,
                min_length=SUMMARY_MIN_LENGTH,
                do_sample=SUMMARY_DO_SAMPLE,
                num_beams=SUMMARY_NUM_BEAMS,
                length_penalty=SUMMARY_LENGTH_PENALTY
            )
        summary = out[0].get('summary_text') or out[0].get('generated_text') or ""
        chunk_summaries.append(summary.strip())

    combined = " ".join(chunk_summaries)
    return combined.strip()

def qa_and_summarize(section_text: str, qa_pipeline, summarizer_pipeline, tokenizer, qa_queries: List[str]) -> str:
    if not section_text or section_text.strip() == "":
        return "No content found for this section."

    final_summary_parts = []
    qa_answers_text = ""
    for query in qa_queries:
        try:
            qa_result = qa_pipeline(question=query, context=section_text)
            if qa_result['score'] > 0.2:
                answer = qa_result['answer'].strip()
                if not answer.endswith(('.', '!', '?')):
                    answer += '.'
                qa_answers_text += f" - {answer}\n"
        except Exception as e:
            print(f" ! QA failed for query '{query}': {e}")

    if qa_answers_text:
        final_summary_parts.append(qa_answers_text)

    try:
        bart_summary = summarize_text_bart(section_text, summarizer_pipeline, tokenizer)
        bart_summary_lines = [line.strip() for line in bart_summary.split('.') if line.strip()]
        final_summary_parts.extend([f" - {line}." for line in bart_summary_lines])
    except Exception as e:
        print(f" ! BART summarization failed: {e}")
        final_summary_parts.append(" - A general summary could not be generated.")

    return "\n".join(final_summary_parts)

def write_summaries_to_pdf(summaries: Dict[str, str], ordered_headers: List, filename: str):
    doc = SimpleDocTemplate(filename, pagesize=letter)
    styles = getSampleStyleSheet()
    justified = ParagraphStyle('Justified', parent=styles['Normal'], alignment=4, fontName="Helvetica", fontSize=11, spaceAfter=8)
    heading = ParagraphStyle('Heading', parent=styles['Heading1'], alignment=0, fontName="Helvetica-Bold", fontSize=14, spaceAfter=12)
    story = []
    for key, _ in ordered_headers:
        if key.startswith('_') or key == 'document_summary': continue
        title = key.replace('_', ' ').title()
        content = summaries.get(key, "")
        story.append(Paragraph(f"--- {title} ---", heading))
        if not content or content.strip() == "No content found for this section.":
            story.append(Paragraph("No content found / summary not available.", justified))
        else:
            paragraphs = content.split('\n')
            for para in paragraphs:
                if para.strip():
                    story.append(Paragraph(para, justified))
        story.append(Spacer(1, 14))
    story.append(Spacer(1, 8))
    story.append(Paragraph(f"Generated by BART model: {BART_MODEL}. Timestamp: {time.ctime()}", styles['Normal']))
    doc.build(story)

def load_pipelines(device):
    tokenizer = BartTokenizer.from_pretrained(BART_MODEL)
    model = BartForConditionalGeneration.from_pretrained(BART_MODEL)
    summarizer = pipeline("summarization", model=model, tokenizer=tokenizer, device=device)
    qa_pipeline = pipeline("question-answering", model=QA_MODEL, tokenizer=QA_MODEL, device=device)
    return summarizer, qa_pipeline, tokenizer

# ---------- MAIN ----------
def main():
    start_time = time.time()
    device = 0 if torch.cuda.is_available() else -1
    try:
        pdf_path = choose_pdf_path()
        print(f"Reading PDF from: {pdf_path}")
        text = pdf_to_text(pdf_path)
    except FileNotFoundError as e:
        print(e)
        return

    print("Extracting sections using regex headers...")
    sections = extract_all_sections_ordered(text, ORDERED_HEADERS)

    try:
        print(f"Loading models on {'GPU' if device == 0 else 'CPU'}...")
        summarizer, qa_pipeline, tokenizer = load_pipelines(device)
        _ = summarizer("Test summarization for CUDA check.", max_length=20, min_length=5)
    except Exception as e:
        print(f"Warning: Encountered error loading/using on GPU: {e}")
        print("Falling back to CPU...")
        device = -1
        summarizer, qa_pipeline, tokenizer = load_pipelines(device)
        print("Device set to use CPU")

    summaries = {}
    print("Summarizing each extracted section with QA-driven BART summarization...")

    for key, content in sections.items():
        print(f" - Processing section: {key} (length={len(content.split())} words)")
        try:
            qa_queries = QUERIES_FOR_QA.get(key, [])
            if (key in ["treatment_arms", "dosing_schedule"]) and not content.strip():
                try:
                    print(f"   ! No content in '{key}' from main PDF. Trying fallback PDF...")
                    fallback_text = pdf_to_text(FALLBACK_PDF_PATH)
                    fallback_sections = extract_all_sections_ordered(fallback_text, ORDERED_HEADERS)
                    fallback_content = fallback_sections.get(key, "")
                    if fallback_content.strip():
                        print(f"   > Successfully extracted '{key}' from fallback PDF.")
                        content = fallback_content
                except Exception as e:
                    print(f"   ! Failed to extract '{key}' from fallback PDF: {e}")

            if not content.strip():
                summaries[key] = "No content found for this section."
                continue

            summary = qa_and_summarize(content, qa_pipeline, summarizer, tokenizer, qa_queries)
            summaries[key] = summary
        except Exception as e:
            print(f"   ! Summarization failed for {key}: {e}. Falling back to extracted raw text (truncated).")
            summaries[key] = (content[:4000] + "...") if len(content) > 4000 else content

    print(f"Writing summarized output to {OUTPUT_PDF} ...")
    out_headers = ORDERED_HEADERS
    write_summaries_to_pdf(summaries, out_headers, OUTPUT_PDF)

    elapsed = time.time() - start_time
    print("\nDone.")
    print(f"Summaries written to: {OUTPUT_PDF}")
    print(f"Elapsed time: {elapsed:.1f} seconds")

if __name__ == "__main__":
    main()